# Dados de cheias do Zenodo (UFRGS/IPH)

Leitura do GeoPackage *Banco de dados das cheias na Região Hidrográfica do Lago Guaíba*, v1.2 —
https://zenodo.org/records/13227745 (CC-BY-4.0), baixado por `scripts/coleta_zenodo.py`.

Perguntas:
1. Quais bairros de Porto Alegre alagaram em 2024 e nas cheias anteriores?
2. A simulação do INCT concorda com as manchas de inundação do Zenodo?
3. A água medida em campo em maio de 2024 bate com a mancha simulada?

Toda alteração feita nos dados está comentada no código com **ALTERAÇÃO**.

## Leitura dos dados

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

PROJECT_DIR = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
ORIGINAIS = PROJECT_DIR / "dados" / "originais"
GPKG = ORIGINAIS / "zenodo" / "cheias_rhguaiba_2024_db_v12.gpkg"

# Bairros de Porto Alegre (camada do GeoPackage)
bairros = gpd.read_file(GPKG, layer="poa_admin_bairros")

# Simulação do INCT em formato longo (uma linha por nível e bairro)
with open(ORIGINAIS / "INCT" / "data.json", encoding="utf-8") as f:
    raw = json.load(f)
inct = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")), "bairro": bairro, **valores}
    for nivel, bs in raw.items()
    for bairro, valores in bs.items()
])
print("Bairros no GeoPackage:", len(bairros), "| CRS:", bairros.crs)
print("Bairros no INCT:", inct["bairro"].nunique())

## Limpeza: nomes dos bairros

O GeoPackage escreve `PASSO DAREIA` e o INCT `Passo D'Areia` (**formatos inconsistentes**).
Criamos uma coluna `chave` com o nome sem acento, sem apóstrofo e em minúsculas para juntar as duas bases.

In [ ]:
def padroniza(nome):
    """Remove acentos e apóstrofos, deixa em minúsculas e com espaços simples."""
    sem_acento = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    sem_apostrofo = sem_acento.replace("'", "")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", sem_apostrofo.lower()).split())

# ALTERAÇÃO: dois bairros têm nome diferente nas duas fontes; trocamos o nome do GeoPackage
# pelo nome usado no INCT (conferido manualmente no mapa de bairros).
NOMES_CORRIGIDOS = {"cel aparicio borges": "coronel aparicio borges", "sao jose": "vila sao jose"}

bairros["chave"] = bairros["nmbop"].map(padroniza).replace(NOMES_CORRIGIDOS)
inct["chave"] = inct["bairro"].map(padroniza)

so_no_inct = sorted(set(inct["chave"]) - set(bairros["chave"]))
print(f"Bairros do INCT sem polígono no GeoPackage ({len(so_no_inct)}):", so_no_inct)

O GeoPackage usa a divisão antiga (83 bairros) e o INCT a nova, de 2016 (94 bairros).
Os 11 bairros acima não têm polígono, então **ficam fora das comparações** (dado faltante: removido).

## Área alagada por bairro em cada cheia

Para cada bairro calculamos a **% da área que fica dentro da mancha de inundação** de cada cheia.
As manchas são simuladas pelos autores sobre o relevo atual:

| Cheia | Nível do Guaíba | Camada |
|---|---|---|
| 1941 | 4,75 m | `poa_inundacao_1941_475cm` |
| 1967 | 3,13 m | `poa_inundacao_1967_313cm` |
| 2015 | 2,94 m | `poa_inundacao_2015_294cm` |
| set/2023 | 3,18 m | `poa_inundacao_2023_set_318cm` |
| nov/2023 | 3,46 m | `poa_inundacao_2023_nov_346cm` |
| 2024 | 5,30 m | `poa_inundacao_2024_530cm` |

In [ ]:
CHEIAS = {
    "1941": "poa_inundacao_1941_475cm",
    "1967": "poa_inundacao_1967_313cm",
    "2015": "poa_inundacao_2015_294cm",
    "set/2023": "poa_inundacao_2023_set_318cm",
    "nov/2023": "poa_inundacao_2023_nov_346cm",
    "2024": "poa_inundacao_2024_530cm",
}
NIVEL = {"1941": 4.75, "1967": 3.13, "2015": 2.94, "set/2023": 3.18, "nov/2023": 3.46, "2024": 5.30}
cheias = list(CHEIAS)

# ALTERAÇÃO: os polígonos dos bairros têm coordenada Z (altitude), que não usamos; removemos com force_2d().
bairros["geometry"] = bairros.force_2d()
bairros["area_km2"] = bairros.area / 1e6  # CRS EPSG:31982 está em metros, então a área sai em m²

def le_mancha(camada):
    """Lê uma mancha e junta todos os polígonos em um só."""
    mancha = gpd.read_file(GPKG, layer=camada).to_crs(bairros.crs)
    # ALTERAÇÃO: make_valid() corrige polígonos inválidos (bordas que se cruzam), senão a interseção dá erro.
    return mancha.make_valid().union_all()

manchas = {cheia: le_mancha(camada) for cheia, camada in CHEIAS.items()}

alag = bairros[["chave", "nmbop", "area_km2", "geometry"]].copy()
for cheia, mancha in manchas.items():
    alag[cheia] = (bairros.intersection(mancha).area / 1e6 / bairros["area_km2"] * 100).round(2)

alag.drop(columns="geometry").sort_values("2024", ascending=False).head(10)

### Mapa das cheias

Cada mapa mostra os bairros coloridos pela % da área alagada. Dá para ver que as cheias pequenas (2015, 2023)
atingem só a orla e as ilhas, enquanto 1941 e 2024 avançam sobre o Centro e a Zona Norte.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, cheia in zip(axes.flat, sorted(cheias, key=NIVEL.get)):
    alag.plot(column=cheia, ax=ax, cmap="Blues", vmin=0, vmax=100, edgecolor="gray", linewidth=0.3)
    ax.set_title(f"{cheia} — {NIVEL[cheia]:.2f} m".replace(".", ","))
    ax.set_axis_off()

fig.colorbar(plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(0, 100)),
             ax=axes, shrink=0.6, label="% da área do bairro alagada")
fig.suptitle("Área alagada por bairro em cada cheia (ordenado pelo nível do Guaíba)", fontsize=14)
plt.show()

### Área alagada total × nível do Guaíba

Soma da área alagada em todos os bairros. Quanto maior o nível, maior a área — mas até a cheia de 2015 (2,94 m)
já mostra ~90 km², porque as manchas incluem as ilhas e áreas úmidas que ficam sob água em qualquer cheia.

In [ ]:
area_total = pd.DataFrame({
    "cheia": cheias,
    "nivel_m": [NIVEL[c] for c in cheias],
    "area_km2": [(alag[c] * alag["area_km2"] / 100).sum() for c in cheias],
}).sort_values("nivel_m")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(area_total["nivel_m"], area_total["area_km2"], marker="o", color="steelblue")
for _, linha in area_total.iterrows():
    ax.annotate(linha["cheia"], (linha["nivel_m"], linha["area_km2"]), textcoords="offset points", xytext=(5, -12))
ax.set_title("Área alagada em Porto Alegre por cheia")
ax.set_xlabel("Nível do Guaíba (m)")
ax.set_ylabel("Área alagada (km²)")
plt.tight_layout()
plt.show()

## Estatística descritiva

Resumo da % alagada por bairro em cada cheia. O pandas calcula **obliquidade** e **curtose** pelas fórmulas
ajustadas (Fisher-Pearson); a curtose já vem como **excesso de curtose** (normal = 0).

In [ ]:
resumo = alag[cheias].describe().T[["mean", "50%", "std", "min", "max"]]
resumo["obliquidade"] = alag[cheias].skew()
resumo["curtose (excesso)"] = alag[cheias].kurt()
resumo.round(2)

### Distribuição da % alagada em 2024

O histograma (com curva suavizada) mostra quais valores aparecem com mais frequência; a curva tracejada é uma
**normal com a mesma média e desvio padrão**, só como referência. O boxplot resume mediana, dispersão e extremos.

In [ ]:
valores = alag["2024"]
media, mediana, desvio = valores.mean(), valores.median(), valores.std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(valores, bins=20, binrange=(0, 100), stat="density", kde=True, ax=axes[0], color="steelblue")
x = np.linspace(-60, 110, 300)
axes[0].plot(x, np.exp(-((x - media) ** 2) / (2 * desvio ** 2)) / (desvio * np.sqrt(2 * np.pi)),
             "--", color="gray", label="normal de referência")
axes[0].axvline(media, color="red", label=f"média = {media:.1f}%")
axes[0].axvline(mediana, color="green", label=f"mediana = {mediana:.1f}%")
axes[0].set_title(f"Distribuição — obliquidade = {valores.skew():.2f}")
axes[0].set_xlabel("% da área do bairro alagada")
axes[0].set_ylabel("Densidade")
axes[0].legend()

sns.boxplot(y=valores, ax=axes[1], color="lightcoral")
axes[1].set_title("Dispersão e valores extremos")
axes[1].set_ylabel("% da área do bairro alagada")

fig.tight_layout()
plt.show()

**Leitura:** a maioria dos bairros tem 0% (áreas altas) e alguns quase 100% (orla) — **assimetria positiva**,
com cauda à direita: a média fica à direita da mediana. A normal de referência não descreve esses dados
(ela até "prevê" porcentagens negativas).

### Valores extremos em cada cheia

Os pontos fora das caixas são bairros muito diferentes dos demais. Aqui eles **não são erro**: são os bairros
da orla e das ilhas (ex.: Arquipélago), então **mantemos** esses valores.

In [ ]:
longo = alag.melt(id_vars="nmbop", value_vars=cheias, var_name="cheia", value_name="pct")

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=longo, x="cheia", y="pct", order=sorted(cheias, key=NIVEL.get), color="lightsteelblue", ax=ax)
ax.set_title("% da área alagada por bairro em cada cheia (ordenado pelo nível)")
ax.set_xlabel("Cheia")
ax.set_ylabel("% da área do bairro alagada")
plt.tight_layout()
plt.show()

## Mancha observada por satélite

As manchas acima são **simuladas**. A camada `rhguaiba_planet-skysat_inundacao_obs_20240506` é a mancha
**observada** em 06/05/2024 (imagens Planet/Skysat + pontos de campo), perto do pico de 5,35 m.

In [ ]:
obs = gpd.read_file(GPKG, layer="rhguaiba_planet-skysat_inundacao_obs_20240506")
# ALTERAÇÃO: a camada está em graus (EPSG:4326); convertemos para o CRS dos bairros (metros)
# e removemos a coordenada Z, que não usamos.
obs = obs.to_crs(bairros.crs)
obs["geometry"] = obs.force_2d()
# ALTERAÇÃO: a camada cobre toda a bacia do Guaíba; recortamos para o retângulo de Porto Alegre
# e corrigimos geometrias inválidas antes de juntar tudo em um polígono.
mancha_obs = gpd.clip(obs.make_valid(), bairros.total_bounds).union_all()

alag["obs 2024"] = (bairros.intersection(mancha_obs).area / 1e6 / bairros["area_km2"] * 100).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, (titulo, mancha) in zip(axes, [("Simulada a 5,30 m", manchas["2024"]),
                                      ("Observada em 06/05/2024", mancha_obs)]):
    bairros.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.4)
    gpd.GeoSeries([mancha], crs=bairros.crs).clip(bairros.union_all()).plot(ax=ax, color="steelblue", alpha=0.7)
    ax.set_title(titulo)
    ax.set_axis_off()
fig.suptitle("Mancha de inundação de 2024 em Porto Alegre", fontsize=14)
plt.tight_layout()
plt.show()

## Comparação com o INCT

Juntamos a % de prédios afetados do INCT a **5,3 m** (nível mais próximo do pico) às manchas de 2024.
O `merge` com `how="inner"` mantém só os 83 bairros presentes nas duas bases.

In [ ]:
inct_53 = inct.loc[inct["nivel_m"] == 5.3, ["chave", "bairro", "percentage"]]
inct_53 = inct_53.rename(columns={"percentage": "INCT 5,3m"})
comp = alag.merge(inct_53, on="chave", how="inner")
print(len(comp), "bairros comparados")

### Mapa: observado × INCT

À esquerda, % da área alagada observada; à direita, % de prédios afetados segundo o INCT. Mesmas cores e escala.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, (coluna, titulo) in zip(axes, [("obs 2024", "Observado 06/05/2024 (% da área)"),
                                       ("INCT 5,3m", "INCT a 5,3 m (% dos prédios)")]):
    comp.plot(column=coluna, ax=ax, cmap="Blues", vmin=0, vmax=100, edgecolor="gray", linewidth=0.3)
    ax.set_title(titulo)
    ax.set_axis_off()
fig.colorbar(plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(0, 100)), ax=axes, shrink=0.6, label="%")
plt.show()

### Correlação de Pearson

A matriz mostra a correlação entre todas as manchas e o INCT. Valores perto de 1 = os bairros que alagam mais
numa fonte também alagam mais na outra. Como Pearson mede só relação **linear**, também olhamos a dispersão.

In [ ]:
colunas = sorted(cheias, key=NIVEL.get) + ["obs 2024", "INCT 5,3m"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(comp[colunas].corr(method="pearson"), annot=True, fmt=".2f", cmap="RdBu", vmin=-1, vmax=1,
            ax=axes[0])
axes[0].set_title("Matriz de correlação de Pearson")

sns.scatterplot(data=comp, x="obs 2024", y="INCT 5,3m", ax=axes[1], color="steelblue", s=60)
axes[1].plot([0, 100], [0, 100], "--", color="gray", label="concordância perfeita")
for _, linha in comp.assign(dif=(comp["INCT 5,3m"] - comp["obs 2024"]).abs()).nlargest(5, "dif").iterrows():
    axes[1].annotate(linha["bairro"], (linha["obs 2024"], linha["INCT 5,3m"]), fontsize=8,
                     textcoords="offset points", xytext=(4, 4))
axes[1].set_title("Cada ponto é um bairro (5 maiores diferenças com nome)")
axes[1].set_xlabel("% da área alagada (observado 06/05/2024)")
axes[1].set_ylabel("% de prédios afetados (INCT 5,3 m)")
axes[1].legend()

fig.tight_layout()
plt.show()

**Leitura:** correlação alta (~0,96): INCT e mancha observada concordam sobre **quais** bairros alagam.
Pontos abaixo da diagonal são bairros em que a água cobriu mais área do que a fração de prédios afetados no INCT
(ex.: áreas alagadas com poucos prédios). Correlação ≠ causalidade: as duas medidas sobem juntas porque dependem
da mesma coisa — a altitude do bairro.

### Diferença por bairro

Diferença em pontos percentuais entre o INCT e o observado, para os bairros atingidos. Barras para a esquerda:
o INCT afeta menos do que a área observada sugere; para a direita: afeta mais.

In [ ]:
comp["diferenca_pp"] = comp["INCT 5,3m"] - comp["obs 2024"]
atingidos = comp[(comp["obs 2024"] > 1) | (comp["INCT 5,3m"] > 1)].sort_values("diferenca_pp")

fig, ax = plt.subplots(figsize=(9, 9))
cores = ["indianred" if d < 0 else "seagreen" for d in atingidos["diferenca_pp"]]
ax.barh(atingidos["bairro"], atingidos["diferenca_pp"], color=cores)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("INCT (5,3 m) − observado (06/05/2024)")
ax.set_xlabel("Diferença (pontos percentuais)")
plt.tight_layout()
plt.show()

## Classes de alagamento (discretização)

Discretização **manual**, com pontos de corte fáceis de explicar:

| Classe | % |
|---|---|
| Sem alagamento | até 1% |
| Baixo | 1% a 25% |
| Médio | 25% a 50% |
| Alto | acima de 50% |

Usamos 1% (e não 0%) porque frações minúsculas vêm de bordas de polígonos encostando no rio.
A tabela cruza a classe observada com a do INCT: a **diagonal** são os bairros em que as duas fontes concordam.

In [ ]:
CORTES = [-0.01, 1, 25, 50, 100]
CLASSES = ["Sem alagamento", "Baixo", "Médio", "Alto"]
comp["classe_obs"] = pd.cut(comp["obs 2024"], bins=CORTES, labels=CLASSES)
comp["classe_inct"] = pd.cut(comp["INCT 5,3m"], bins=CORTES, labels=CLASSES)

tabela = pd.crosstab(comp["classe_obs"], comp["classe_inct"])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(tabela, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Quantidade de bairros por classe")
ax.set_xlabel("INCT a 5,3 m")
ax.set_ylabel("Observado 06/05/2024")
plt.tight_layout()
plt.show()
print(f"Concordância: {np.trace(tabela.values)} de {tabela.values.sum()} bairros na mesma classe")

## Quais bairros alagam sempre? (binarização + Jaccard)

**Binarização por limiar:** o bairro "alagou" (1) se mais de 1% da área ficou sob água; senão 0.

O gráfico da esquerda conta em quantas das 6 cheias cada bairro alagou. O da direita compara as cheias com o
**coeficiente de Jaccard**: o 0 ("não alagou") não interessa (binário **assimétrico**), então ele mede só a
proporção de bairros alagados que as duas cheias têm em comum.

In [ ]:
LIMIAR = 1  # %
binario = (comp[colunas] > LIMIAR).astype(int)
binario["n_cheias"] = binario[cheias].sum(axis=1)

def jaccard(a, b):
    m11 = ((a == 1) & (b == 1)).sum()
    m10 = ((a == 1) & (b == 0)).sum()
    m01 = ((a == 0) & (b == 1)).sum()
    return m11 / (m01 + m10 + m11)

sim = pd.DataFrame([[jaccard(binario[a], binario[b]) for b in colunas] for a in colunas],
                   index=colunas, columns=colunas)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
contagem = binario["n_cheias"].value_counts().reindex(range(7), fill_value=0)
axes[0].bar(contagem.index, contagem.values, color="darkslateblue")
for x, y in zip(contagem.index, contagem.values):
    axes[0].text(x, y + 0.5, str(y), ha="center")
axes[0].set_title("Em quantas das 6 cheias cada bairro alagou")
axes[0].set_xlabel("Número de cheias")
axes[0].set_ylabel("Bairros")

sns.heatmap(sim, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Similaridade de Jaccard entre os bairros alagados")
fig.tight_layout()
plt.show()

recentes = ["1967", "2015", "set/2023", "nov/2023"]
novos = comp.loc[(binario["2024"] == 1) & (binario[recentes].sum(axis=1) == 0), "bairro"].tolist()
print("Alagaram em 2024 mas em nenhuma cheia recente (1967, 2015, 2023):", novos)

## Pontos de nível medidos em campo

Camada `poa_pontos_nivel`: pontos onde a equipe do GPDEN/IPH/UFRGS marcou **a borda da água** em maio de 2024,
com o nível do Guaíba no dia (`Nivel_G_m`). Primeiro conferimos tipos, faltantes e duplicatas.

In [ ]:
pontos = gpd.read_file(GPKG, layer="poa_pontos_nivel")
pontos.info()
print("\nLinhas duplicadas:", pontos.duplicated(subset=["X", "Y", "Data"]).sum())

In [ ]:
# ALTERAÇÃO: Nivel_G_m vem como texto; convertemos para número.
pontos["Nivel_G_m"] = pontos["Nivel_G_m"].astype(float)
# ALTERAÇÃO: os pontos estão em graus (EPSG:4326); convertemos para o CRS dos bairros (metros)
# para medir distâncias, e removemos a coordenada Z.
pontos = pontos.to_crs(bairros.crs)
pontos["geometry"] = pontos.force_2d()
pontos["dia"] = pontos["Data"] + " (" + pontos["Nivel_G_m"].astype(str) + " m)"

# Mancha com nível mais próximo do nível medido em cada dia
MANCHA_DO_NIVEL = {
    4.65: "poa_inundacao_1941_475cm",      # 4,75 m
    4.95: "poa_inundacao_simulada_500cm",  # 5,00 m
    5.06: "poa_inundacao_simulada_500cm",  # 5,00 m
    5.29: "poa_inundacao_2024_530cm",      # 5,30 m
    5.33: "poa_inundacao_2024_530cm",      # 5,30 m
}
pontos["mancha"] = pontos["Nivel_G_m"].map(MANCHA_DO_NIVEL)

# Distância euclidiana (em metros) de cada ponto até a borda da mancha
for camada in pontos["mancha"].unique():
    mancha = le_mancha(camada)
    linhas = pontos["mancha"] == camada
    pontos.loc[linhas, "dentro_da_mancha"] = pontos.loc[linhas].within(mancha)
    pontos.loc[linhas, "dist_borda_m"] = pontos.loc[linhas].distance(mancha.boundary)

pontos.groupby("dia").agg(pontos=("dist_borda_m", "size"),
                          pct_dentro=("dentro_da_mancha", lambda s: round(s.mean() * 100)),
                          dist_mediana_m=("dist_borda_m", "median")).round(1)

### Onde a água foi medida

Os pontos sobre a mancha simulada de 5,30 m. Os pontos são a borda real da água; se a simulação estiver boa,
eles ficam perto da borda azul.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 11))
bairros.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.4)
gpd.GeoSeries([manchas["2024"]], crs=bairros.crs).clip(bairros.union_all()).plot(ax=ax, color="lightsteelblue")
sns.scatterplot(x=pontos.geometry.x, y=pontos.geometry.y, hue=pontos["dia"], hue_order=sorted(pontos["dia"].unique()), palette="tab10", s=25, ax=ax)
ax.set_title("Borda da água medida em campo × mancha simulada a 5,30 m")
ax.legend(title="Data (nível)", loc="lower left")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Distância até a borda simulada

Cada ponto é comparado com a mancha de nível mais próximo do dia (5,29–5,33 m → mancha de 5,30 m;
4,95–5,06 m → 5,00 m; 4,65 m → 4,75 m). Quanto menor a distância, mais a simulação acerta até onde a água chegou.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ordem = sorted(pontos["dia"].unique())
sns.boxplot(data=pontos, x="dia", y="dist_borda_m", order=ordem, color="lightcoral", ax=ax, showfliers=False)
sns.stripplot(data=pontos, x="dia", y="dist_borda_m", order=ordem, hue="dentro_da_mancha",
              palette={True: "steelblue", False: "darkorange"}, size=4, ax=ax)
ax.set_title("Distância entre a borda medida em campo e a borda simulada")
ax.set_xlabel("Data (nível do Guaíba)")
ax.set_ylabel("Distância (m)")
ax.legend(title="Dentro da mancha?")
plt.tight_layout()
plt.show()

**Leitura:**
* 05 e 06/05 (5,29–5,33 m): quase todos os pontos ficam dentro da mancha de 5,30 m, perto da borda.
* 08/05 (5,06 m, Sarandi): quase todos ficam **fora** da mancha de 5,00 m — a água real foi além da simulada.
* 12/05 (4,65 m): comparado com a mancha de 4,75 m (nível maior), então é esperado que fiquem dentro.

## Limitações
* As manchas históricas e a de 5,30 m são **simuladas**; só a do satélite (06/05/2024) é observada.
* % de área alagada e % de prédios afetados (INCT) são medidas diferentes: compare a tendência, não o valor.
* 11 bairros novos do INCT não têm polígono no GeoPackage e ficaram fora.